In [14]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
#import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
#from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from skimage.feature import hog


from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB


In [15]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Model

### Reading Data

In [ ]:

CSV_PATH = "../archive/styles.csv"      
IMAGE_DIR = "../archive/images"         # folder containing image files
IMG_SIZE = (128, 128)        # resize all images to same size
df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
df = df[["id", "masterCategory"]].dropna()
df["id"] = df["id"].astype(str)
df["image_path"] = df["id"].apply(lambda x: os.path.join(IMAGE_DIR, f"{x}.jpg"))
df = df[df["image_path"].apply(os.path.exists)].copy()
print("Total usable rows:", len(df))
print("\nClass distribution:")
print(df["masterCategory"].value_counts())

Total usable rows: 44182

Class distribution:
masterCategory
Apparel           21307
Accessories       11198
Footwear           9143
Personal Care      2403
Free Items          105
Sporting Goods       25
Home                  1
Name: count, dtype: int64


In [3]:
# Filter
valid_classes = df["masterCategory"].value_counts()
valid_classes = valid_classes[valid_classes >= 50].index
df = df[df["masterCategory"].isin(valid_classes)].copy()

print("\nAfter filtering rare classes:")
print(df["masterCategory"].value_counts())


After filtering rare classes:
masterCategory
Apparel          21307
Accessories      11198
Footwear          9143
Personal Care     2403
Free Items         105
Name: count, dtype: int64


In [4]:
# Load Images
def load_images_and_labels(dataframe, img_size=(128, 128)):
    X = []
    y = []

    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        img_path = row["image_path"]
        label = row["masterCategory"]

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, img_size)

        X.append(img)
        y.append(label)

    return np.array(X), np.array(y)

X_images, y = load_images_and_labels(df, IMG_SIZE)

print("Loaded image array shape:", X_images.shape)
print("Loaded labels shape:", y.shape)

100%|██████████| 44156/44156 [07:03<00:00, 104.36it/s]


Loaded image array shape: (44156, 128, 128, 3)
Loaded labels shape: (44156,)


In [5]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("\nClasses:", list(label_encoder.classes_))


Classes: [np.str_('Accessories'), np.str_('Apparel'), np.str_('Footwear'), np.str_('Free Items'), np.str_('Personal Care')]


In [6]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_images,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (35324, 128, 128, 3)
Test shape: (8832, 128, 128, 3)


### Raw pixel values

In [7]:
class RawPixelTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, resize_shape=(64, 64)):
        self.resize_shape = resize_shape

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for img in X:
            resized = cv2.resize(img, self.resize_shape)
            features.append(resized.flatten())
        return np.array(features)


IMG_SIZE_REDUCED = (64, 64)

kfold = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


models = {
    "PCA + Logistic Regression": Pipeline([
        ("features", RawPixelTransformer(resize_shape=IMG_SIZE_REDUCED)),
        ("scaler", StandardScaler()),
        ("dim_reduce", PCA(n_components=50, random_state=42)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),

    "PCA + Linear SVC": Pipeline([
        ("features", RawPixelTransformer(resize_shape=IMG_SIZE_REDUCED)),
        ("scaler", StandardScaler()),
        ("dim_reduce", PCA(n_components=50, random_state=42)),
        ("clf", LinearSVC(random_state=42, max_iter=3000))
    ]),

    "PCA + Gaussian NB": Pipeline([
        ("features", RawPixelTransformer(resize_shape=IMG_SIZE_REDUCED)),
        ("scaler", StandardScaler()),
        ("dim_reduce", PCA(n_components=50, random_state=42)),
        ("clf", GaussianNB())
    ])
}


results = []
best_model = None
best_score = 0
best_name = ""

for name, model in models.items():
    print(f"\nRunning: {name}")

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kfold,
        scoring="accuracy",
        n_jobs=-1
    )

    mean_score = scores.mean()

    results.append({
        "Model": name,
        "CV Accuracy Mean": mean_score,
        "CV Accuracy Std": scores.std()
    })

    if mean_score > best_score:
        best_score = mean_score
        best_model = model
        best_name = name


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="CV Accuracy Mean", ascending=False)

print("\nModel Comparison:")
print(results_df)


print(f"\nBest Model: {best_name}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

print("\nTraining best model on full training data...")
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("\nTest Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Running: PCA + Logistic Regression

Running: PCA + Linear SVC

Running: PCA + Gaussian NB

Model Comparison:
                       Model  CV Accuracy Mean  CV Accuracy Std
0  PCA + Logistic Regression          0.931548         0.001118
1           PCA + Linear SVC          0.929057         0.000886
2          PCA + Gaussian NB          0.606896         0.006052

Best Model: PCA + Logistic Regression
Best Cross-Validation Accuracy: 0.9315

Training best model on full training data...


c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Test Accuracy:
0.935348731884058

Classification Report:
               precision    recall  f1-score   support

  Accessories       0.87      0.92      0.89      2240
      Apparel       0.96      0.95      0.96      4262
     Footwear       0.98      0.98      0.98      1829
   Free Items       0.00      0.00      0.00        21
Personal Care       0.85      0.73      0.79       480

     accuracy                           0.94      8832
    macro avg       0.73      0.72      0.72      8832
 weighted avg       0.93      0.94      0.93      8832


Confusion Matrix:
[[2059  130   19    0   32]
 [ 173 4054    7    1   27]
 [  26    5 1797    0    1]
 [  18    3    0    0    0]
 [  95   29    5    0  351]]


### HOG (Histogram of Oriented Gradients)

In [8]:
# HOG Transformer with lower memory usage
class HOGTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        resize_shape=(64, 64),
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    ):
        self.resize_shape = resize_shape
        self.orientations = orientations
        self.pixels_per_cell = pixels_per_cell
        self.cells_per_block = cells_per_block
        self.block_norm = block_norm

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []

        for img in X:
            img = cv2.resize(img, self.resize_shape)

            if len(img.shape) == 3:
                gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            else:
                gray = img

            hog_features = hog(
                gray,
                orientations=self.orientations,
                pixels_per_cell=self.pixels_per_cell,
                cells_per_block=self.cells_per_block,
                block_norm=self.block_norm,
                feature_vector=True
            )

            features.append(hog_features.astype(np.float32))

        return np.asarray(features, dtype=np.float32)


HOG_IMG_SIZE = (64, 64)

kfold = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


models = {
    "HOG + Logistic Regression": Pipeline([
        ("hog", HOGTransformer(resize_shape=HOG_IMG_SIZE)),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),

    "HOG + Linear SVC": Pipeline([
        ("hog", HOGTransformer(resize_shape=HOG_IMG_SIZE)),
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(random_state=42, max_iter=3000))
    ]),

    "HOG + Gaussian NB": Pipeline([
        ("hog", HOGTransformer(resize_shape=HOG_IMG_SIZE)),
        ("clf", GaussianNB())
    ])
}


results = []
best_model = None
best_score = 0
best_name = ""

for name, model in models.items():
    print(f"\nRunning: {name}")

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kfold,
        scoring="accuracy",
        n_jobs=1
    )

    mean_score = scores.mean()

    results.append({
        "Model": name,
        "CV Accuracy Mean": mean_score,
        "CV Accuracy Std": scores.std()
    })

    if mean_score > best_score:
        best_score = mean_score
        best_model = model
        best_name = name


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="CV Accuracy Mean", ascending=False)

print("\nModel Comparison:")
print(results_df)


print(f"\nBest Model: {best_name}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

print("\nTraining best model on full training data...")
best_model.fit(X_train, y_train)

print("\nPredicting...")
y_pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Running: HOG + Logistic Regression

Running: HOG + Linear SVC


c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



Running: HOG + Gaussian NB

Model Comparison:
                       Model  CV Accuracy Mean  CV Accuracy Std
0  HOG + Logistic Regression          0.962858         0.001210
1           HOG + Linear SVC          0.953261         0.001670
2          HOG + Gaussian NB          0.722880         0.007272

Best Model: HOG + Logistic Regression
Best Cross-Validation Accuracy: 0.9629

Training best model on full training data...

Predicting...

Test Accuracy: 0.9615036231884058

Classification Report:

               precision    recall  f1-score   support

  Accessories       0.93      0.94      0.93      2240
      Apparel       0.97      0.97      0.97      4262
     Footwear       0.99      0.99      0.99      1829
   Free Items       0.08      0.10      0.09        21
Personal Care       0.93      0.92      0.92       480

     accuracy                           0.96      8832
    macro avg       0.78      0.78      0.78      8832
 weighted avg       0.96      0.96      0.96      8832



### ORB (Oriented FAST and Rotated BRIEF)

In [9]:
class ORBTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, resize_shape=(64, 64), n_keypoints=50):
        self.resize_shape = resize_shape
        self.n_keypoints = n_keypoints

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        orb = cv2.ORB_create(nfeatures=self.n_keypoints)
        features = []

        for img in tqdm(X, desc="Extracting ORB features"):
            resized = cv2.resize(img, self.resize_shape)
            gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)

            keypoints, descriptors = orb.detectAndCompute(gray, None)

            if descriptors is None:
                descriptors = np.zeros((self.n_keypoints, 32), dtype=np.float32)
            else:
                descriptors = descriptors.astype(np.float32)

            if len(descriptors) < self.n_keypoints:
                pad = np.zeros((self.n_keypoints - len(descriptors), 32), dtype=np.float32)
                descriptors = np.vstack((descriptors, pad))
            else:
                descriptors = descriptors[:self.n_keypoints]

            features.append(descriptors.flatten())

        return np.asarray(features, dtype=np.float32)


ORB_IMG_SIZE = (64, 64)

kfold = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

models = {
    "ORB + Logistic Regression": Pipeline([
        ("features", ORBTransformer(resize_shape=ORB_IMG_SIZE, n_keypoints=50)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),

    "ORB + Linear SVC": Pipeline([
        ("features", ORBTransformer(resize_shape=ORB_IMG_SIZE, n_keypoints=50)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", LinearSVC(random_state=42, max_iter=3000))
    ]),

    "ORB + Gaussian NB": Pipeline([
        ("features", ORBTransformer(resize_shape=ORB_IMG_SIZE, n_keypoints=50)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", GaussianNB())
    ])
}


results = []
best_model = None
best_score = 0
best_name = ""

for name, model in models.items():
    print(f"\nRunning: {name}")

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kfold,
        scoring="accuracy",
        n_jobs=1
    )

    mean_score = scores.mean()

    results.append({
        "Model": name,
        "CV Accuracy Mean": mean_score,
        "CV Accuracy Std": scores.std()
    })

    if mean_score > best_score:
        best_score = mean_score
        best_model = model
        best_name = name


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="CV Accuracy Mean", ascending=False)

print("\nModel Comparison:")
print(results_df)

print(f"\nBest Model: {best_name}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

print("\nTraining best model on full training data...")
best_model.fit(X_train, y_train)

print("\nPredicting...")
y_pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Running: ORB + Logistic Regression


Extracting ORB features: 100%|██████████| 11774/11774 [00:01<00:00, 6371.66it/s]



Running: ORB + Linear SVC


Extracting ORB features: 100%|██████████| 11774/11774 [00:01<00:00, 6209.05it/s]



Running: ORB + Gaussian NB


...

Extracting ORB features: 100%|██████████| 35324/35324 [00:06<00:00, 5819.17it/s]



Predicting...


Extracting ORB features: 100%|██████████| 8832/8832 [00:01<00:00, 5851.74it/s]


Test Accuracy: 0.49377264492753625

Classification Report:

               precision    recall  f1-score   support

  Accessories       0.48      0.04      0.07      2240
      Apparel       0.49      0.98      0.66      4262
     Footwear       0.54      0.06      0.11      1829
   Free Items       0.00      0.00      0.00        21
Personal Care       0.30      0.01      0.01       480

     accuracy                           0.49      8832
    macro avg       0.36      0.22      0.17      8832
 weighted avg       0.49      0.49      0.36      8832


Confusion Matrix:

[[  80 2115   42    0    3]
 [  53 4166   41    0    2]
 [  24 1691  112    0    2]
 [   2   17    2    0    0]
 [   6  460   11    0    3]]



c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

### Color Histogram

In [10]:
class ColorHistogramTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, resize_shape=(64, 64), bins=(8, 8, 8)):
        self.resize_shape = resize_shape
        self.bins = bins

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []

        for img in tqdm(X, desc="Extracting Color Histogram features"):
            resized = cv2.resize(img, self.resize_shape)

            # Convert RGB image to HSV color space
            hsv = cv2.cvtColor(resized, cv2.COLOR_RGB2HSV)

            # Compute 3D color histogram
            hist = cv2.calcHist(
                [hsv],
                [0, 1, 2],
                None,
                self.bins,
                [0, 180, 0, 256, 0, 256]
            )

            # Normalize histogram
            hist = cv2.normalize(hist, hist).flatten()

            features.append(hist.astype(np.float32))

        return np.asarray(features, dtype=np.float32)


COLOR_IMG_SIZE = (64, 64)

kfold = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


models = {
    "Color Histogram + Logistic Regression": Pipeline([
        ("features", ColorHistogramTransformer(
            resize_shape=COLOR_IMG_SIZE,
            bins=(8, 8, 8)
        )),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),

    "Color Histogram + Linear SVC": Pipeline([
        ("features", ColorHistogramTransformer(
            resize_shape=COLOR_IMG_SIZE,
            bins=(8, 8, 8)
        )),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", LinearSVC(random_state=42, max_iter=3000))
    ]),

    "Color Histogram + Gaussian NB": Pipeline([
        ("features", ColorHistogramTransformer(
            resize_shape=COLOR_IMG_SIZE,
            bins=(8, 8, 8)
        )),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", GaussianNB())
    ])
}


results = []
best_model = None
best_score = 0
best_name = ""

for name, model in models.items():
    print(f"\nRunning: {name}")

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kfold,
        scoring="accuracy",
        n_jobs=1
    )

    mean_score = scores.mean()

    results.append({
        "Model": name,
        "CV Accuracy Mean": mean_score,
        "CV Accuracy Std": scores.std()
    })

    if mean_score > best_score:
        best_score = mean_score
        best_model = model
        best_name = name


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="CV Accuracy Mean", ascending=False)

print("\nModel Comparison:")
print(results_df)

print(f"\nBest Model: {best_name}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

print("\nTraining best model on full training data...")
best_model.fit(X_train, y_train)

print("\nPredicting...")
y_pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Running: Color Histogram + Logistic Regression


Extracting Color Histogram features: 100%|██████████| 11774/11774 [00:00<00:00, 36322.97it/s]



Running: Color Histogram + Linear SVC


Extracting Color Histogram features: 100%|██████████| 11774/11774 [00:00<00:00, 39638.59it/s]



Running: Color Histogram + Gaussian NB


...

Extracting Color Histogram features: 100%|██████████| 35324/35324 [00:00<00:00, 42438.12it/s]



Predicting...


Extracting Color Histogram features: 100%|██████████| 8832/8832 [00:00<00:00, 35285.73it/s]



Test Accuracy: 0.6268115942028986

Classification Report:

               precision    recall  f1-score   support

  Accessories       0.44      0.31      0.36      2240
      Apparel       0.79      0.90      0.84      4262
     Footwear       0.42      0.56      0.48      1829
   Free Items       0.00      0.00      0.00        21
Personal Care       0.60      0.01      0.01       480

     accuracy                           0.63      8832
    macro avg       0.45      0.35      0.34      8832
 weighted avg       0.61      0.63      0.60      8832


Confusion Matrix:

[[ 691  659  889    0    1]
 [ 248 3824  189    0    1]
 [ 553  258 1018    0    0]
 [   8    7    6    0    0]
 [  67   70  340    0    3]]


c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roshn\anaconda3\envs\cv_project_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

### MobileNetV2

In [ ]:
# MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(128, 128, 3)
)

base_model.trainable = False



def extract_mobilenet_embeddings(images, batch_size=32):
    processed_images = []

    for img in tqdm(images, desc="Preprocessing images"):
        img = cv2.resize(img, (128, 128))

        if img.dtype != np.uint8:
            img = img.astype(np.uint8)

        processed_images.append(img)

    processed_images = np.asarray(processed_images, dtype=np.float32)

    # MobileNetV2 preprocessing
    processed_images = preprocess_input(processed_images)

    embeddings = base_model.predict(
        processed_images,
        batch_size=batch_size,
        verbose=1
    )

    return embeddings


print("\nExtracting train embeddings...")
X_train_embed = extract_mobilenet_embeddings(X_train, batch_size=32)

print("\nExtracting test embeddings...")
X_test_embed = extract_mobilenet_embeddings(X_test, batch_size=32)



kfold = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)



models_dict = {
    "MobileNetV2 Embeddings + Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),

    "MobileNetV2 Embeddings + Linear SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", LinearSVC(random_state=42, max_iter=3000))
    ]),

    "MobileNetV2 Embeddings + Gaussian NB": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", GaussianNB())
    ])
}



results = []
best_model = None
best_score = 0
best_name = ""

for name, model in models_dict.items():
    print(f"\nRunning: {name}")

    scores = cross_val_score(
        model,
        X_train_embed,
        y_train,
        cv=kfold,
        scoring="accuracy",
        n_jobs=1
    )

    mean_score = scores.mean()

    results.append({
        "Model": name,
        "CV Accuracy Mean": mean_score,
        "CV Accuracy Std": scores.std()
    })

    if mean_score > best_score:
        best_score = mean_score
        best_model = model
        best_name = name


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="CV Accuracy Mean", ascending=False)

print("\nModel Comparison:")
print(results_df)



print(f"\nBest Model: {best_name}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

print("\nTraining best model on full training embeddings...")
best_model.fit(X_train_embed, y_train)

print("\nPredicting...")
y_pred = best_model.predict(X_test_embed)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step

Extracting train embeddings...


Preprocessing images: 100%|██████████| 35324/35324 [00:00<00:00, 39757.75it/s]


1104/1104 ━━━━━━━━━━━━━━━━━━━━ 221s 198ms/step

Extracting test embeddings...


Preprocessing images: 100%|██████████| 8832/8832 [00:02<00:00, 3372.12it/s]


276/276 ━━━━━━━━━━━━━━━━━━━━ 71s 257ms/step

Running: MobileNetV2 Embeddings + Logistic Regression

Running: MobileNetV2 Embeddings + Linear SVC

Running: MobileNetV2 Embeddings + Gaussian NB

Model Comparison:
                                          Model  CV Accuracy Mean  \
0  MobileNetV2 Embeddings + Logistic Regression          0.974352   
1           MobileNetV2 Embeddings + Linear SVC          0.971181   
2          MobileNetV2 Embeddings + Gaussian NB          0.931689   

   CV Accuracy Std  
0         0.000423  
1         0.000760  
2         0.000631  

Best Model: MobileNetV2 Embeddings + Logistic Regression
Best Cross-Validation Accuracy: 0.9744

Training best model on full training embeddings...

Predicting...

Test Accuracy: 0.9755434782608695

Classification Report:

               precision    recall  f1-score   support

  Accessories       0.95      0.96      0.96      2240
      Apparel       0.98      0.99      0.98      4262
     Footwear       1.00      1.00    